# [M7] DỰ ÁN AI THẬT ĐỂ ĐƯA VÀO PORTFOLIO
## Buổi S02: EDA Chuyên Sâu, Data Pipeline & Xây Dựng Baseline Sạch

---

### Metadata Khóa Học
* **Module:** M7 — DỰ ÁN AI THẬT ĐỂ ĐƯA VÀO PORTFOLIO
* **Buổi:** S02
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook Preprocessing Pipeline hoàn chỉnh và Clean Baseline Model
* **Mức độ liên quan VOAI / Olympic AI:** Tiêu chuẩn xây dựng Pipeline dữ liệu sạch cấp độ sản phẩm và mốc đối sánh ban đầu.

#### Mục tiêu học tập (Learning Objectives)
- Thực hiện Khám phá dữ liệu chuyên sâu (In-depth EDA) cho bài toán Final Project: Phân tích phân phối, tương quan và phát hiện ngoại lai.
- Xây dựng một Pipeline Tiền xử lý dữ liệu hoàn chỉnh, tự động hóa và có khả năng đóng gói (Modular Data Preprocessing Pipeline).
- Thiết lập mô hình cơ sở sạch sẽ (Clean Baseline Model) đầu tiên cho dự án Portfolio.
- Đo lường và ghi nhận mốc hiệu năng cơ sở (Baseline Performance Benchmark) làm cột mốc định lượng cho các cải tiến tiếp theo.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M7_S01 (Đã khởi tạo Repository và kiểm toán dữ liệu).
- Nắm vững kỹ năng EDA từ Module M1 và xây dựng Baseline từ Module M2.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Sau khi đã có bài toán và dữ liệu thô sạch sẽ từ buổi 1, cám dỗ lớn nhất của một sinh viên non kinh nghiệm là ngay lập tức lao vào tải các mô hình phức tạp về chạy thử. Đó là con đường ngắn nhất dẫn đến thất bại!

Một Kỹ sư AI đẳng cấp luôn bắt đầu bằng việc **lắng nghe dữ liệu**: Khám phá các mối quan hệ tiềm ẩn, tìm ra quy luật phân phối và xây dựng một **Pipeline Tiền xử lý dữ liệu khép kín, mô-đun hóa**. Đồng thời, việc dựng một mô hình Baseline sạch sẽ ngay tại buổi hôm nay sẽ cung cấp cho bạn một thước đo khoa học để chứng minh giá trị của toàn bộ những cải tiến phức tạp ở các buổi sau.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Nguyên tắc Xây dựng Preprocessing Pipeline mô-đun hóa
* Toàn bộ các bước: Điền khuyết (Imputation), Mã hóa danh mục (Encoding), Chuẩn hóa số học (Scaling) phải được đóng gói thành một đối tượng Pipeline duy nhất (ví dụ sử dụng `sklearn.pipeline.Pipeline` hoặc `ColumnTransformer`).
* **Không rò rỉ dữ liệu:** `pipeline.fit(X_train)` học các tham số thống kê chỉ từ tập Train, sau đó áp dụng sang tập Validation/Test.
* Có thể xuất bản (Serialize) toàn bộ pipeline thành file nhị phân (`preprocessor.joblib`) để sẵn sàng phục vụ cho ứng dụng Demo ở Buổi 4!

### 2. Thiết lập Clean Baseline cho Final Project
* Baseline phải là mô hình đơn giản nhất, dễ giải thích nhất nhưng hoạt động ổn định:
  - Phân loại: `DummyClassifier` kết hợp `LogisticRegression` hoặc `DecisionTreeClassifier(max_depth=3)`.
  - Hồi quy: `DummyRegressor` kết hợp `LinearRegression`.
* Ghi lại đầy đủ: Train Score, Validation Score (Accuracy, Macro-F1, RMSE) và thời gian thực thi.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Kiến trúc Preprocessing Pipeline & Baseline Benchmark:
[Dữ liệu thô: X_train, y_train]
              │
              ▼
    [ColumnTransformer Pipeline]
    ├── Cột Số Học: Imputer (Median) ──> StandardScaler()
    └── Cột Danh Mục: Imputer (Mode) ──> OneHotEncoder(handle_unknown='ignore')
              │
              ▼
[Tập Đặc Trưng Đã Chuẩn Hóa] ──> [Huấn luyện Clean Baseline Model] ──> [BASELINE SCORE MỐC]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. KHOI TAO TAP DU LIEU NGHIEP VU CHO FINAL PROJECT (HEALTHCARE PREDICTION)
np.random.seed(42)
N = 600

data = {
    'Age': np.random.randint(20, 80, size=N),
    'Blood_Pressure': np.random.normal(130, 18, size=N).round(1),
    'Cholesterol': np.random.normal(210, 35, size=N).round(1),
    'Smoking_Status': np.random.choice(['Never', 'Former', 'Current', np.nan], size=N, p=[0.4, 0.3, 0.2, 0.1]),
    'Heart_Disease': np.random.choice([0, 1], size=N, p=[0.75, 0.25]) # Target (25% Duong tinh)
}

df_project = pd.DataFrame(data)

# Chia Train / Test doc lap ngay tu dau de chong Leakage
X = df_project.drop(columns=['Heart_Disease'])
y = df_project['Heart_Disease']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Tap Train: {len(X_train)} mau | Tap Test: {len(X_test)} mau")

### 🔍 Code Walkthrough (Khối 1)
Khối code thiết lập bài toán Final Project với đầy đủ đặc trưng hỗn hợp:
- Dữ liệu gồm cả biến số học (`Age`, `Blood_Pressure`, `Cholesterol`) và biến danh mục có missing value (`Smoking_Status`).
- Phân chia Train/Test ngay từ đầu có Stratification bảo đảm tính nguyên vẹn khoa học.


In [ ]:
# 2. XAY DUNG MODULAR PREPROCESSING PIPELINE BANG COLUMNTRANSFORMER
num_cols = ['Age', 'Blood_Pressure', 'Cholesterol']
cat_cols = ['Smoking_Status']

# Pipeline con cho bien so hoc
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline con cho bien danh muc
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Hop nhat thanh Preprocessor tong the
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# 3. HUAN LUYEN VA DANH GIA CLEAN BASELINE MODEL
baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(random_state=42))
])

# Huan luyen toan bo luong chi voi 1 lenh fit duy nhat!
baseline_pipeline.fit(X_train, y_train)

# Du doan va Danh gia tren tap Test doc lap
test_preds = baseline_pipeline.predict(X_test)
base_acc = accuracy_score(y_test, test_preds)
base_f1 = f1_score(y_test, test_preds, average='macro')

print("===== KET QUA CLEAN BASELINE PERFORMANCE (MILESTONE) =====")
print(f"Baseline Model: Logistic Regression with Scaled & One-Hot Features")
print(f"Test Accuracy : {base_acc*100:.2f}%")
print(f"Test Macro-F1 : {base_f1:.4f}")
print("-> MOC CHUAN: Moi mo hinh nang cao o buoi M7_S03 bat buoc phai vuot qua diem Macro-F1 nay!")

### 🔍 Code Walkthrough (Khối 2)
Khối code xây dựng một Pipeline Scikit-Learn chuẩn mực cấp độ sản phẩm:
- `ColumnTransformer`: Tự động áp dụng tiền xử lý chuyên biệt cho từng loại cột dữ liệu.
- Đóng gói toàn bộ từ tiền xử lý đến mô hình vào `baseline_pipeline`.
- Đo lường và thiết lập cột mốc điểm số Macro-F1 chính thức cho dự án.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Lưu trữ Preprocessor đã huấn luyện
Sử dụng thư viện `joblib` để lưu đối tượng `baseline_pipeline` vào file `portfolio_project/models/baseline_pipeline.joblib`. Viết đoạn code nạp lại file và chạy thử dự đoán trên 1 mẫu mới.

### Bài 2 (Easy): So sánh với Dummy Classifier
Huấn luyện một `DummyClassifier(strategy='most_frequent')` trên cùng tập dữ liệu. Tính điểm Macro-F1 và xác nhận xem mô hình Logistic Regression của bạn đã đánh bại được Dummy Classifier bao nhiêu phần trăm.

### Bài 3 (Medium): Bổ sung biểu đồ phân tích EDA vào thư mục báo cáo
Viết mã nguồn vẽ biểu đồ phân phối điểm số tương quan giữa `Blood_Pressure` và `Cholesterol` theo nhãn bệnh `Heart_Disease`. Tự động lưu biểu đồ chất lượng cao (DPI=300) vào đường dẫn `portfolio_project/reports/figures/eda_correlation.png`.

### Bài 4 (Medium): Kiểm tra khả năng xử lý danh mục lạ (Unseen Categories)
Tạo một mẫu dữ liệu mới có giá trị `Smoking_Status = 'Occasional'` (giá trị chưa từng xuất hiện trong tập Train). Chạy qua `baseline_pipeline` và giải thích tại sao tham số `handle_unknown='ignore'` trong `OneHotEncoder` giúp mô hình không bị sập (crash).

### Bài 5 (Challenge): Viết Custom Scikit-Learn Transformer
Tự viết một class `OutlierCapper(lower_quantile=0.01, upper_quantile=0.99)` kế thừa từ `BaseEstimator` và `TransformerMixin` của Scikit-Learn. Tích hợp Transformer này vào đầu của `num_pipeline` để tự động cắt tỉa các điểm dị biệt.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong các buổi bảo vệ dự án tốt nghiệp hoặc phỏng vấn tuyển dụng AI, nhà tuyển dụng sẽ đánh giá rất cao việc bạn sử dụng **`sklearn.pipeline.Pipeline` thay vì viết các đoạn code tiền xử lý rời rạc**. Một pipeline duy nhất nhận dữ liệu thô và xuất ra dự đoán thể hiện tư duy đóng gói sản phẩm sạch sẽ, không có lỗi rò rỉ dữ liệu và sẵn sàng đưa lên môi trường Server thực tế.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh tiền xử lý dữ liệu bằng cách gọi `preprocessor.fit_transform()` riêng cho tập Train và `preprocessor.fit_transform()` riêng cho tập Test, dẫn đến số lượng cột của hai tập bị lệch nhau do tập Test thiếu một giá trị danh mục:

```python
# CODE CÓ LỖI LỆCH CỘT DO FIT LẠI TẬP TEST:
# X_train_clean = preprocessor.fit_transform(X_train) # Sinh ra 6 cot
# X_test_clean = preprocessor.fit_transform(X_test)   # Chi sinh ra 5 cot vi thieu nhan!
# ValueError: Number of features of the model must match the input!
```

**Nhiệm vụ của bạn:**
1. Khẳng định quy tắc cốt tử: **Tuyệt đối KHÔNG BAO GIỜ gọi `fit` hoặc `fit_transform` trên tập Test!**
2. Sử dụng cấu trúc `Pipeline` như đã làm trong code cell trên để ngăn chặn hoàn toàn lỗi này.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Tại sao trong một dự án AI chuyên nghiệp, chúng ta nên đóng gói toàn bộ quy trình tiền xử lý và mô hình vào một đối tượng `Pipeline` duy nhất?
   - A. Để máy tính chạy tốn nhiều RAM hơn.
   - B. Giúp quy trình xử lý dữ liệu trở nên mô-đun hóa, triệt tiêu hoàn toàn nguy cơ rò rỉ dữ liệu (Data Leakage) và dễ dàng đóng gói để triển khai lên sản phẩm thực tế chỉ bằng một hàm `predict()`.
   - C. Vì Scikit-Learn cấm viết code thông thường.
   - D. Để giấu mã nguồn không cho người khác xem.

2. Tham số `handle_unknown='ignore'` trong `OneHotEncoder` có tác dụng gì khi đưa vào triển khai thực tế?
   - A. Tự động xóa các hàng dữ liệu lạ.
   - B. Khi gặp một giá trị danh mục mới chưa từng xuất hiện trong tập huấn luyện (ví dụ một thành phố lạ), bộ mã hóa sẽ gán toàn bộ các cột one-hot tương ứng bằng số 0 thay vì báo lỗi sập chương trình.
   - C. Biến giá trị đó thành số âm.
   - D. Đổi tên cột dữ liệu.

3. Trong buổi M7_S02, việc thiết lập mốc "Clean Baseline Performance" có mục đích gì?
   - A. Để kết thúc dự án sớm.
   - B. Thiết lập chuẩn đối sánh tối thiểu có cơ sở khoa học để đánh giá xem các mô hình Deep Learning hoặc kỹ thuật tinh chỉnh phức tạp ở buổi sau có thực sự mang lại hiệu quả vượt trội hay không.
   - C. Để nộp báo cáo cho ban giám khảo chấm điểm ngay.
   - D. Để giảm thời gian huấn luyện.

4. Khi chuẩn hóa các cột số học (Scaling), tại sao giá trị trung bình $\mu$ và độ lệch chuẩn $\sigma$ chỉ được học từ tập Train?
   - A. Vì tập Train có nhiều mẫu hơn.
   - B. Để bảo đảm tính trung thực tuyệt đối: Mô hình không được phép biết trước phân phối thống kê của dữ liệu kiểm tra tương lai.
   - C. Vì tập Test không có số học.
   - D. Do giới hạn của thuật toán.

5. File định dạng `.joblib` hoặc `.pkl` thường được sử dụng trong hệ sinh thái Python AI để làm gì?
   - A. Lưu trữ file video demo.
   - B. Tuần tự hóa (Serialize) và lưu trữ trạng thái của các đối tượng mô hình và pipeline tiền xử lý đã huấn luyện xuống đĩa cứng để tái sử dụng.
   - C. Soạn thảo văn bản báo cáo.
   - D. Nén mã nguồn Python.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Quy trình EDA chuyên sâu và phân tích tương quan cho Final Project.
  - Cách xây dựng `ColumnTransformer` và `Pipeline` khép kín cấp độ sản xuất.
  - Kỹ thuật phòng ngừa lệch số lượng cột với `handle_unknown='ignore'`.
  - Thiết lập cột mốc hiệu năng Clean Baseline chính thức cho dự án.
* **What I can now do:**
  - Tự tin đóng gói mọi quy trình tiền xử lý phức tạp thành một đối tượng duy nhất.
  - Đo lường và lưu trữ mô hình Baseline sẵn sàng cho giai đoạn bứt phá hiệu năng.
* **What I still need to practice:**
  - Nâng cấp mô hình, tinh chỉnh siêu tham số và thực hiện **Ablation Study** ở Buổi 3.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M7_S03**, chúng ta sẽ bước vào giai đoạn tăng tốc hiệu năng: **Thử Nghiệm Mô Hình Nâng Cao, Tinh Chỉnh Siêu Tham Số & Ablation Study** — Đưa Random Forest, Gradient Boosting và Deep MLP vào so găng, tinh chỉnh siêu tham số và đo lường đóng góp của từng thành phần!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M7**:
- Khám phá dữ liệu chuyên sâu (In-depth EDA) & Data storytelling ✓
- Xây dựng Modular Preprocessing Pipeline với ColumnTransformer ✓
- Thiết lập Clean Baseline model & Benchmark mốc hiệu năng ban đầu ✓

* **Session Deliverable:** Mã nguồn Modular Data Preprocessing Pipeline + Clean Baseline Benchmark
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
